# UJIAN TENGAH SEMESTER (UTS) - THREAT HUNTING & CYBER THREAT INTELLIGENCE
## Program Studi S1 Teknik Komputer | Fakultas Ilmu Komputer | Universitas Amikom Yogyakarta

**Nama Mahasiswa:** Dzul Ikram A.S. Sambaiyang  
**NIM:** 24.83.1130  
**Email Amikom:** ikram.sambaiyang19@students.amikom.ac.id  
**Mata Kuliah:** Threat Hunting & Cyber Threat Intelligence (THCTI)  
**Dataset Telemetri:** [witfoo/precinct6-cybersecurity](https://huggingface.co/datasets/witfoo/precinct6-cybersecurity)  
**Repository GitHub:** [https://github.com/Dzul-Ikram/UTS-ThreatHunting](https://github.com/Dzul-Ikram/UTS-ThreatHunting)  
**Google Colab Notebook:** [https://colab.research.google.com/drive/1ys4l468E6lEuRepVZIOlKEUzR-kNK1LJ](https://colab.research.google.com/drive/1ys4l468E6lEuRepVZIOlKEUzR-kNK1LJ)  
**Live Interactive Dashboard:** [https://dzul-ikram.github.io/UTS-ThreatHunting/](https://dzul-ikram.github.io/UTS-ThreatHunting/)  

---
### Deskripsi Kasus & Metodologi Investigasi
Notebook ini menyajikan implementasi **Hypothesis-Based Threat Hunting** berbasis data telemetri riil Security Operations Center (SOC) enterprise dari dataset **WitFoo Precinct 6** (2.250.185 sinyal, 51.371 insiden). Investigasi difokuskan pada kampanye serangan terorganisir **Data Theft (Partisi 233678)** pada tenant **ORG-0003**, dengan insiden kunci **`25512200-01d5-11ee-8c41-374f8f3a6a1c` (Defiant Pike 233678)** yang memicu skor kecurigaan tertinggi (**0.859 / 85.9%**).

Investigasi ini disusun untuk memenuhi secara komprehensif seluruh rubrik penilaian ujian (Total: 100 Poin):
1. **Analisis Landscape & Threat Actor (Bobot: 15 Poin)**
2. **Hipotesis dan Metodologi Hunting (Bobot: 15 Poin)**
3. **Telemetry Correlation & Evidence Timeline (Bobot: 20 Poin)**
4. **Klasifikasi IOC, IOA, TTP & Pemetaan MITRE ATT&CK (Bobot: 20 Poin)**
5. **Hunting Query dan Detection Engineering (Bobot: 20 Poin)**
6. **Hunting Report dan Link Demo (Bobot: 10 Poin)**

## 0. Inisialisasi Environment & Pemuatan Data Telemetri Forensik
Sel ini secara otomatis memverifikasi environment (Google Colab atau sistem lokal) dan mengunduh berkas telemetri kampanye (`campaign_telemetry.json`) langsung dari repositori GitHub.

In [ ]:
import os
import sys
import json
import urllib.request
from datetime import datetime, timezone

try:
    import pandas as pd
except ImportError:
    pd = None

try:
    import matplotlib.pyplot as plt
    HAS_MATPLOTLIB = True
except ImportError:
    HAS_MATPLOTLIB = False

try:
    from IPython.display import display
except ImportError:
    display = print

# Tentukan sumber data telemetri kampanye
DATA_FILE = 'campaign_telemetry.json'
GITHUB_RAW_URL = 'https://raw.githubusercontent.com/Dzul-Ikram/UTS-ThreatHunting/main/campaign_telemetry.json'

if not os.path.exists(DATA_FILE):
    print(f"Mengunduh {DATA_FILE} dari repositori GitHub...")
    try:
        urllib.request.urlretrieve(GITHUB_RAW_URL, DATA_FILE)
        print("Unduhan dataset forensik berhasil!")
    except Exception as e:
        print(f"Gagal mengunduh: {e}")
else:
    print(f"Berkas lokal {DATA_FILE} terdeteksi.")

# Load dataset telemetri kampanye Partisi 233678
with open(DATA_FILE, 'r', encoding='utf-8') as f:
    campaign_incidents = json.load(f)

print(f"[STATUS] Berhasil memuat {len(campaign_incidents)} insiden terkoordinasi dalam Kampanye Partisi 233678.")

## 1. Analisis Landscape & Attack Lifecycle (Soal 1 - Bobot: 15 Poin)
### 1.1 Profiling Threat Actor, Motivasi, Kemampuan, dan Target Aset
Berdasarkan analisis telemetri tenant **ORG-0003**, teridentifikasi aktivitas kelompok kejahatan siber yang menyasar kompromi perimeter nirkabel dan workstation resepsionis untuk upaya eksfiltrasi data korporat.

In [ ]:
# Profiling Landscape Ancaman
landscape_profile = {
    "Komponen": [
        "Threat Actor",
        "Motivasi",
        "Kemampuan (Capability)",
        "Target Aset",
        "Attack Lifecycle"
    ],
    "Karakteristik & Analisis Evidence": [
        "Financially Motivated Cybercrime Group / Initial Access Broker (IAB). Karakteristik sindikat kejahatan siber terorganisir yang menyasar kompromi perimeter web dan workstation enterprise untuk pencurian data dan monetisasi akses.",
        "Keuntungan finansial (Financial Gain) melalui pencurian data sensitif (Data Theft), penjualan akses ke dark web, dan penguasaan server korban.",
        "Tingkat Menengah ke Atas (Capable/Advanced). Mampu melakukan komunikasi C2 terenkripsi (HTTPS port 443), teknik penghindaran firewall port non-standar (13731, 8880, 4112, 4010), dan pemindaian lateral terkoordinasi antar cabang.",
        "Aset korporat ORG-0003: Workstation resepsionis 192.168.96.104 di kantor pusat (NORHQ_WAP_RECEPTION) dan switch/gateway cabang Atlanta, Conyers, Loxley, Northbrook.",
        "1. Initial Access (T1190) -> 2. Execution (T1059/Web Shell) -> 3. Discovery (T1046) -> 4. Lateral Movement (T1021) -> 5. Exfiltration (T1041/T1567)."
    ]
}

df_landscape = pd.DataFrame(landscape_profile)
pd.set_option('display.max_colwidth', None)
display(df_landscape)

### 1.2 Diferensiasi Tegas: Fakta vs Asumsi vs Hipotesis
Pemisahan objektif antara data empiris yang tercatat pada log telemetri, deduksi logis analis, dan proposisi investigasi yang diuji.

In [ ]:
# Tabel Diferensiasi Fakta, Asumsi, dan Hipotesis
fha_data = {
    "Kategori": ["FAKTA (Observed Truth)", "ASUMSI (Logical Deduction)", "HIPOTESIS (Investigative Proposal)"],
    "Pernyataan Analisis": [
        "Host 192.168.96.104 menginisiasi sesi TCP keluar ke 100.64.43.120:443 pada 2023-06-02 21:19:28 UTC. Sesi koneksi diblokir (action: deny) oleh sensor NORHQ_WAP_RECEPTION. FireEye HX memicu 'FireEye HX Domain Match' dan Stealthwatch memicu 'High Concern Index Alarm'. Terjadi pemindaian port 13731 di cabang Loxley (29 leads, 15 node).",
        "IP 100.64.43.120 adalah representasi tersanitasi dari server C2 penyerang (HMAC RFC 6598). Muatan sesi port 443 mengandung data korporat yang telah dikumpulkan oleh malware/web shell.",
        "H1: Host resepsionis terinfeksi malware yang mengeksekusi C2 beaconing periodik via HTTPS.\nH2: Aktivitas port 13731 di cabang Loxley merupakan pemindaian internal lateral reconnaissance sebelum exfiltrasi."
    ],
    "Dasar Bukti / Keterbatasan": [
        "Tercatat eksplisit pada 4 triggering leads insiden 25512200 dan log syslog NORHQ_WAP_RECEPTION.",
        "Dideduksi dari klasifikasi Modus Operandi 'Data Theft' dan standar sanitasi dataset.",
        "Proposisi investigatif yang diuji menggunakan korelasi telemetri EDR, NDR, dan firewall."
    ]
}

df_fha = pd.DataFrame(fha_data)
display(df_fha)

## 2. Perumusan Threat Hunting Hypothesis (Soal 2 - Bobot: 15 Poin)
Perumusan dua hipotesis berbobot tinggi lengkap dengan tujuan investigasi, ruang lingkup host/user/time window, data yang dibutuhkan, langkah penyelidikan, kriteria validasi, dan tindakan lanjutan.

In [ ]:
hypotheses_data = {
    "Parameter": [
        "Pernyataan Hipotesis",
        "Tujuan Investigasi",
        "Ruang Lingkup (Scope)",
        "Data yang Dibutuhkan",
        "Langkah Penyelidikan",
        "Kriteria Validasi",
        "Tindakan Lanjutan (Action Plan)",
        "Status Validasi"
    ],
    "Hipotesis 1 (H1: Outbound C2 Beaconing & Exfiltration)": [
        "Terdapat host internal di segmen resepsionis/WAP kantor pusat (NORHQ) yang mengalami kompromi dan secara aktif melakukan komunikasi outbound tidak wajar ke domain C2 berbahaya via protokol HTTPS (port 443) untuk melakukan eksfiltrasi data rahasia.",
        "Mendeteksi kanal komunikasi C2 aktif, mengidentifikasi proses inisiator pada sistem operasi host, memvalidasi domain IOC, dan memastikan integritas data korporat.",
        "Host: 192.168.96.104 (MAC: 00:A5:54:CB:19:84) | Time Window: 2023-06-02 20:00:00 s/d 23:59:59 UTC.",
        "Network Flow Logs (Meraki WAP, Cisco ASA), EDR Process & Domain Match Logs (FireEye HX), Cisco Stealthwatch HCI alarms.",
        "1) Filter flow log keluar ke 100.64.43.120:443; 2) Ekstrak nama domain yang memicu FireEye HX Domain Match; 3) Periksa process tree di host; 4) Evaluasi volume transfer byte pada Stealthwatch.",
        "SUPPORTED bila terbukti korelasi domain match EDR + alarm lonjakan HCI + deny log firewall. REJECTED bila koneksi terbukti update software resmi vendor. INCONCLUSIVE bila log EDR tidak tersedia.",
        "Jika Terbukti: Isolasi host dari jaringan, sinkhole domain C2 pada DNS internal, cabut kredensial aktif, lakukan memory dump triage. Jika Terbantahkan: Lakukan tuning signature EDR.",
        "SUPPORTED (Tervalidasi oleh Evidence)"
    ],
    "Hipotesis 2 (H2: Multi-Branch Lateral Reconnaissance)": [
        "Penyerang memanfaatkan host perantara di jaringan cabang (Loxley/Atlanta/Conyers) untuk melakukan pemindaian horizontal dan probing lateral movement pada port non-standar (13731, 8880, 4112, 4010) secara terkoordinasi.",
        "Memetakan penyebaran infeksi lateral antar kantor cabang dan mendeteksi aktivitas pemindaian port (service discovery) sebelum eksfiltrasi dilakukan.",
        "Host cabang: 100.64.41.160 (Loxley), 100.64.27.241 (Atlanta) | Time Window: 2023-06-02 18:00:00 s/d 2023-06-03 02:00:00 UTC.",
        "Cisco Stealthwatch Flow Logs, Meraki L7 & Firewall deny alarms, Windows Event 5156.",
        "1) Ekstraksi event network flow pada port anomali (13731, 8880, 4112, 4010); 2) Hitung jumlah distinct destination IP per host asal; 3) Evaluasi alarm penimbunan data (data hoarding) Stealthwatch.",
        "SUPPORTED bila terbukti 1 host memindai >10 target dalam waktu singkat tanpa tujuan bisnis sah. REJECTED bila port terbukti software backup terdistribusi resmi. INCONCLUSIVE bila log flow lokal tidak lengkap.",
        "Jika Terbukti: Terapkan microsegmentation antar cabang, blokir port 13731/8880 pada router ACL. Jika Terbantahkan: Daftarkan port aplikasi ke dalam CMDB.",
        "SUPPORTED (Tervalidasi oleh Evidence)"
    ]
}

df_hypotheses = pd.DataFrame(hypotheses_data)
display(df_hypotheses)

## 3. Telemetry Correlation & Evidence Timeline (Soal 3 - Bobot: 20 Poin)
### 3.1 Identifikasi Sumber Telemetri Forensik
Korelasi dilakukan menggunakan minimal dua sumber data independen yang saling melengkapi:
1. **Endpoint Detection and Response (EDR):** `FireEye Endpoint Security (HX Series)` — mendeteksi kecocokan domain jahat (`FireEye HX Domain Match`) pada sistem operasi host.
2. **Network Detection and Response (NDR):** `Cisco Stealthwatch` — mendeteksi anomali perilaku trafik jaringan (`High Concern Index Alarm` dan akumulasi data host).
3. **Perimeter Network & Wireless Security:** `Cisco Meraki WAP (NORHQ_WAP_RECEPTION)` — mencatat sesi TCP keluar port 64813 -> 443 dengan aksi pemblokiran (*action: deny*).

In [ ]:
# Ekstraksi seluruh lead event dari insiden kampanye untuk menyusun Evidence Timeline Kronologis
timeline_records = []

for inc in campaign_incidents:
    for l in inc.get('leads', []):
        ts = l.get('observed_at')
        ts_utc = datetime.fromtimestamp(ts, tz=timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC') if ts else 'N/A'
        src = f"{l.get('clientip')}:{l.get('clientport')}"
        dst = f"{l.get('serverip')}:{l.get('serverport')}"
        timeline_records.append({
            "Timestamp (UTC)": ts_utc,
            "Epoch": ts,
            "Source Entity": src,
            "Destination Entity": dst,
            "Activity / Protocol": f"{str(l.get('protocol', '')).upper()} ({str(l.get('action', '')).upper()})",
            "Detection Rule / Lead": l.get('description'),
            "Sensor Source": l.get('product')
        })

df_timeline_all = pd.DataFrame(timeline_records)
df_timeline_all = df_timeline_all.sort_values(by='Epoch').drop_duplicates(subset=['Epoch', 'Source Entity', 'Destination Entity', 'Detection Rule / Lead'])

print(f"Total chronological evidence events extracted: {len(df_timeline_all)}")
display(df_timeline_all[['Timestamp (UTC)', 'Source Entity', 'Destination Entity', 'Activity / Protocol', 'Detection Rule / Lead', 'Sensor Source']])

In [ ]:
# Visualisasi Sebaran Skor Kecurigaan dan Sensor Multi-Sumber Kampanye
if HAS_MATPLOTLIB:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    inc_names = [i.get('name') for i in campaign_incidents]
    inc_scores = [i.get('suspicion_score', 0) for i in campaign_incidents]
    axes[0].barh(inc_names, inc_scores, color='#1e3a8a')
    axes[0].set_xlabel('Suspicion Score (0.0 - 1.0)')
    axes[0].set_title('Tingkat Kecurigaan Insiden dalam Kampanye 233678 (Tenant ORG-0003)')
    axes[0].set_xlim(0, 1.0)
    axes[0].grid(axis='x', linestyle='--', alpha=0.5)
    sensor_counts = df_timeline_all['Sensor Source'].value_counts() if pd is not None else {}
    if len(sensor_counts) > 0:
        axes[1].pie(sensor_counts, labels=sensor_counts.index, autopct='%1.1f%%', colors=['#1e3a8a', '#0284c7', '#0d9488', '#d97706'])
        axes[1].set_title('Distribusi Sumber Telemetri Forensik (Multi-Sensor)')
    plt.tight_layout()
    plt.show()
else:
    print('Matplotlib tidak terpasang di runtime ini. Menampilkan ringkasan teks:')
    for i in campaign_incidents:
        print(f"- {i.get('name')}: Skor {i.get('suspicion_score')}")


## 4. Klasifikasi IOC, IOA, TTP & Pemetaan MITRE ATT&CK (Soal 4 - Bobot: 20 Poin)
### 4.1 Klasifikasi Temuan: IOC, IOA, dan TTP
Berikut adalah klasifikasi forensik komprehensif atas temuan investigasi:

In [ ]:
# Tabel Klasifikasi IOC, IOA, dan TTP
indicators_classification = {
    "Klasifikasi": [
        "IOC (Indicator of Compromise)",
        "IOC (Indicator of Compromise)",
        "IOC (Indicator of Compromise)",
        "IOA (Indicator of Attack)",
        "IOA (Indicator of Attack)",
        "IOA (Indicator of Attack)",
        "TTP (Tactics, Techniques, Procedures)",
        "TTP (Tactics, Techniques, Procedures)",
        "TTP (Tactics, Techniques, Procedures)"
    ],
    "Entitas / Parameter": [
        "External C2 IP: 100.64.43.120 (RFC 6598 sanitasi)",
        "Compromised Host: 192.168.96.104 (MAC: 00:a5:54:cb:19:84)",
        "Anomalous Destination Ports: 13731, 8880, 4112, 4010",
        "Koneksi Outbound HTTPS mencurigakan dari segmen WAP resepsionis",
        "Pemindaian port horizontal serentak ke 15 node jaringan cabang Loxley",
        "Disrupsi otomatis firewall perimeter (action: deny) pada traffic keluar",
        "T1190 - Initial foothold melalui eksploitasi web perimeter",
        "T1071.001 & T1041 - C2 beaconing & data exfiltration via HTTPS",
        "T1046 & T1021 - Internal network discovery & lateral movement"
    ],
    "Keterangan & Signifikansi Forensik": [
        "Target komunikasi C2 yang terdeteksi cocok dengan blacklist domain FireEye HX.",
        "Workstation resepsionis kantor pusat yang menjadi ground-zero infeksi.",
        "Port non-standar yang digunakan aktor untuk probing lateral antar cabang.",
        "Pola komunikasi beaconing periodik yang memicu Stealthwatch HCI Alarm.",
        "Aktivitas pengintaian aktif (active reconnaissance) sebelum eksekusi data theft.",
        "Bukti intervensi kontrol keamanan yang memutus kebocoran data keluar.",
        "Vektor masuk awal penyerang ke dalam ekosistem korporat ORG-0003.",
        "Protokol terenkripsi yang dipilih untuk mengaburkan inspeksi paket inspeksi.",
        "Metode pergerakan menyamping untuk menguasai segmen jaringan cabang lain."
    ]
}

df_indicators = pd.DataFrame(indicators_classification)
display(df_indicators)

### 4.2 Pemetaan Mendalam MITRE ATT&CK Matrix
Pemetaan perilaku ke taktik dan teknik MITRE ATT&CK Enterprise Matrix dengan alasan teknis dan tingkat keyakinan (*Confidence Level*):

In [ ]:
# Pemetaan MITRE ATT&CK Matrix
mitre_mapping = {
    "MITRE Tactic": [
        "Initial Access (TA0001)",
        "Command and Control (TA0011)",
        "Exfiltration (TA0010)",
        "Exfiltration (TA0010)",
        "Discovery (TA0007)",
        "Lateral Movement (TA0008)"
    ],
    "Technique & ID": [
        "T1190 - Exploit Public-Facing Application",
        "T1071.001 - Web Protocols (HTTPS/443)",
        "T1041 - Exfiltration Over C2 Channel",
        "T1567 - Exfiltration Over Web Service",
        "T1046 - Network Service Discovery",
        "T1021 - Remote Services"
    ],
    "Alasan Pemetaan Berdasarkan Evidence": [
        "Eksploitasi aplikasi web perimeter (sesuai modul kompromi web server & relasi WAF Barracuda) untuk penanaman foothold awal.",
        "Host internal 192.168.96.104 menghubungi IP eksternal 100.64.43.120 via port 443 yang terbukti cocok dengan blacklist C2 domain pada FireEye HX.",
        "Modus operandi diklasifikasikan sebagai Data Theft dengan alarm anomali penimbunan data keluar pada Cisco Stealthwatch.",
        "Upaya transmisi data keluar menggunakan format web request terenkripsi untuk mengelabui filter perimeter firewall.",
        "Host di cabang Loxley (100.64.41.160) memindai 15 target internal pada port anomali 13731 (29 detection leads).",
        "Percobaan pergerakan lateral internal ke gateway Atlanta (port 8880) dan Conyers (port 4112)."
    ],
    "Confidence Level": ["Medium", "High", "High", "High", "High", "Medium"]
}

df_mitre = pd.DataFrame(mitre_mapping)
display(df_mitre)

### 4.3 Batasan Atribusi (Attribution Limitations)
Dalam melakukan analisis ancaman, penting untuk mencatat batasan teknis atribusi berikut:
1. **Sanitasi IP (RFC 6598):** Seluruh IP publik telah disanitasi menggunakan skema HMAC konsisten ke blok CGNAT (`100.64.0.0/10`), sehingga resolusi geolokasi presisi (AS Number / Negara C2) tidak dapat dipastikan 100%.
2. **Enkripsi TLS 1.3:** Lalu lintas port 443 terenkripsi end-to-end tanpa SSL Decryption appliance pada gateway WAP, sehingga muatan file spesifik yang dicoba dieksfiltrasi tidak dapat didekripsi dari paket mentah.
3. **Ketiadaan File Hash Biner:** Data network syslog murni tidak menyertakan hash SHA256 proses biner yang berjalan, sehingga atribusi ke unit APT spesifik dibatasi pada tingkat profil taktis (*Cybercrime IAB*).

## 5. Threat Hunting Queries & Detection Engineering (Soal 5 - Bobot: 20 Poin)
Menyajikan minimal dua hunting query menggunakan sintaks platform SIEM standar (Splunk SPL / EQL) serta implementasi kueri reproduksi berbasis Python/SQL, lengkap dengan analisis False Positive dan rekomendasi tuning deteksi.

### 5.1 Kueri 1: Cross-Source Temporal Join (EDR Alert + Firewall Deny Outbound)
```spl
# SPLUNK SPL SYNTAX:
index=fireeye sourcetype="fe_hx_alerts" signature="*Domain Match*"
| eval match_time=_time, target_c2=server_ip, host_ip=client_ip
| join type=inner max=0 host_ip [
    search index=meraki sourcetype="meraki:security" action="deny" dest_port=443
    | eval deny_time=_time, dest_c2=dst_ip
]
| where abs(match_time - deny_time) <= 300 AND target_c2 == dest_c2
| table _time, host_ip, target_c2, signature, action, dest_port
```
- **Field yang dicari:** `client_ip`, `server_ip`, `signature`, `action`, `dest_port`, `_time`.
- **Logika Filter & Korelasi:** Menggabungkan alert EDR Domain Match dengan event firewall deny pada port 443 dalam jendela waktu $\le 300$ detik pada IP tujuan yang sama.
- **Kemungkinan False Positive:** Update berkala browser/antivirus yang salah teridentifikasi pada blacklist usang.
- **Ide Tuning Deteksi:** Whitelist domain resmi cloud provider dan tambahkan threshold minimum byte transfer.

In [ ]:
# SIMULASI EKSEKUSI KUERI 1 BERBASIS PYTHON DATA ENGINE (KORELASI EDR + NDR + FIREWALL)
def execute_hunting_query_1(incidents):
    alerts = []
    for inc in incidents:
        tools = [t.get('name') for t in inc.get('tools', {}).values()]
        leads = inc.get('leads', [])
        
        has_edr_domain = any('domain match' in str(l.get('description')).lower() for l in leads)
        has_firewall_deny = any(l.get('action') == 'deny' or l.get('action') == 'block' for l in leads)
        has_stealthwatch = any('stealthwatch' in str(t).lower() for t in tools)
        
        if has_edr_domain and has_firewall_deny and has_stealthwatch:
            target_host = [n.get('ip') for n in inc.get('nodes', {}).values() if n.get('internal')]
            target_c2 = [n.get('ip') for n in inc.get('nodes', {}).values() if not n.get('internal')]
            alerts.append({
                "Incident ID": inc.get('id'),
                "Incident Name": inc.get('name'),
                "Suspicion Score": inc.get('suspicion_score'),
                "Internal Infected Host": target_host[0] if target_host else 'N/A',
                "External C2 Target": target_c2[0] if target_c2 else 'N/A',
                "Correlated Sensors": list(set(tools)),
                "Status": "CONFIRMED THREAT (C2 Exfiltration Disrupted)"
            })
    return pd.DataFrame(alerts)

df_q1_results = execute_hunting_query_1(campaign_incidents)
print("[QUERY 1 RESULTS - Cross-Source Temporal Join]")
display(df_q1_results)

### 5.2 Kueri 2: Multi-Branch Lateral Probing & Port Sweep Detector
```spl
# SPLUNK SPL SYNTAX:
index=stealthwatch sourcetype="cisco:stealthwatch:flows" dest_port IN (13731, 8880, 4112, 4010)
| bin _time span=10m
| stats dc(dest_ip) as distinct_targets, count as total_probes, values(dest_port) as targeted_ports by _time, src_ip, sensor_name
| where distinct_targets >= 5
| sort - distinct_targets
```
- **Field yang dicari:** `src_ip`, `dest_ip`, `dest_port`, `sensor_name`, `_time`.
- **Logika Filter & Korelasi:** Agregasi host internal yang menghubungi $\ge 5$ host tujuan berbeda pada port anomali dalam jendela waktu 10 menit.
- **Kemungkinan False Positive:** Scanner kerentanan resmi (Vulnerability Scanner seperti Nessus/Qualys) atau agen backup terdistribusi.
- **Ide Tuning Deteksi:** Kecualikan subnet/IP scanner resmi internal dan pasang alerting jika port sweep diikuti traffic transfer volume besar.

In [ ]:
# SIMULASI EKSEKUSI KUERI 2 BERBASIS PYTHON DATA ENGINE (LATERAL PROBING DETECTOR)
def execute_hunting_query_2(timeline_df):
    anomalous_ports = ['13731', '8880', '4112', '4010']
    sweeps = []
    for idx, row in timeline_df.iterrows():
        dst_port = row['Destination Entity'].split(':')[-1] if ':' in row['Destination Entity'] else ''
        src_ip = row['Source Entity'].split(':')[0]
        dst_ip = row['Destination Entity'].split(':')[0]
        if dst_port in anomalous_ports:
            sweeps.append({
                "src_ip": src_ip,
                "dst_ip": dst_ip,
                "dst_port": dst_port,
                "sensor": row['Sensor Source']
            })
            
    df_sweeps = pd.DataFrame(sweeps)
    if not df_sweeps.empty:
        summary = df_sweeps.groupby(['src_ip', 'dst_port', 'sensor']).agg(
            probed_hosts=('dst_ip', 'nunique'),
            probe_attempts=('dst_ip', 'count')
        ).reset_index()
        return summary
    return pd.DataFrame()

df_q2_results = execute_hunting_query_2(df_timeline_all)
print("[QUERY 2 RESULTS - Lateral Probing Aggregation]")
display(df_q2_results)

## 6. Laporan Singkat, Rekomendasi Deteksi & Bukti Demo (Soal 6 - Bobot: 10 Poin)
### 6.1 Executive Summary & Temuan Utama
- **Executive Summary:** Berdasarkan perburuan ancaman (*threat hunting*) pada telemetri tenant **ORG-0003**, ditemukan kampanye terkoordinasi **Data Theft (Partisi 233678)** yang memanfaatkan workstation resepsionis kantor pusat (`192.168.96.104`) sebagai basis kompromi awal untuk upaya transmisi C2 keluar dan pergerakan lateral ke kantor-kantor cabang (Atlanta, Conyers, Loxley, Northbrook).
- **Metodologi:** Investigasi menerapkan kerangka kerja *Hypothesis-Based Threat Hunting* yang dikombinasikan dengan korelasi multi-sensor (*Diamond Model for Intrusion Analysis* dan *MITRE ATT&CK Framework*).
- **Temuan Utama:** Insiden puncak `25512200-01d5-11ee-8c41-374f8f3a6a1c` (Defiant Pike 233678, skor 0.859) berhasil mengaitkan *FireEye HX Domain Match* dengan *Cisco Stealthwatch High Concern Index Alarm* dan aksi pemblokiran *Cisco Meraki WAP* pada port HTTPS 443.
- **Tingkat Keyakinan (Confidence Level):** **HIGH (Tinggi)** untuk keberadaan infeksi host dan aktivitas C2 beaconing; **MEDIUM** untuk atribuisi kelompok penyerang spesifik karena enkripsi TLS dan sanitasi IP RFC 6598.
- **Dampak Potensial:** Kebocoran data rahasia korporat, kompromi jaringan cabang nirkabel, dan potensi instalasi ransomware atau web shell persisten.

In [ ]:
# Matriks Prioritas Tindak Lanjut Terprioritaskan (Action Plan P1, P2, P3)
action_plan = {
    "Prioritas": ["P1 (Immediate - 0 s/d 24 Jam)", "P2 (Short-Term - 1 s/d 3 Hari)", "P3 (Long-Term - 1 s/d 2 Minggu)"],
    "Tingkat": ["Kritis", "Tinggi", "Menengah"],
    "Tindakan Remediasi & Rekomendasi Deteksi": [
        "1. Isolasi jaringan secara logis terhadap host 192.168.96.104.\n2. Blokir IP 100.64.43.120 pada seluruh perimeter firewall dan sinkhole DNS.\n3. Reset seluruh kredensial akun yang pernah logon pada workstation terinfeksi.",
        "1. Tutup port non-standar (13731, 8880, 4112, 4010) pada ACL router cabang.\n2. Terapkan microsegmentation: isolasi subnet nirkabel resepsionis dari segmen server.\n3. Ambil forensic disk/memory dump untuk analisis artefak web shell/backdoor.",
        "1. Deploy kueri korelasi otomatis SIEM (EDR match + outbound deny flow).\n2. Terapkan TLS Inspection pada perimeter gateway untuk inspeksi muatan port 443.\n3. Lakukan audit integritas direktori web server berkala untuk mencegah SEO spam."
    ],
    "Penanggung Jawab": ["Tim SOC & Incident Response", "Tim Network & Security Engineer", "Tim Security Architect"]
}

df_plan = pd.DataFrame(action_plan)
display(df_plan)

### 6.2 Lampiran Link Demo & Verifikasi Analisis (Sesuai Ketentuan Soal 6)
Sesuai ketentuan Soal 6, seluruh proses pencarian event, korelasi multi-sumber, penyusunan timeline, dan dasar pengambilan keputusan analisis dapat diakses dan diverifikasi secara langsung melalui tautan publik berikut:

1. **Google Colab Notebook (Live):**  
   [https://colab.research.google.com/drive/1ys4l468E6lEuRepVZIOlKEUzR-kNK1LJ](https://colab.research.google.com/drive/1ys4l468E6lEuRepVZIOlKEUzR-kNK1LJ)  
   *(Alternatif 1-Click Run langsung dari GitHub: [Buka di Google Colab](https://colab.research.google.com/github/Dzul-Ikram/UTS-ThreatHunting/blob/main/Threat_Hunting_Investigation.ipynb))*

2. **Repository GitHub Resmi:**  
   [https://github.com/Dzul-Ikram/UTS-ThreatHunting](https://github.com/Dzul-Ikram/UTS-ThreatHunting)

3. **Live Interactive Web Dashboard (GitHub Pages):**  
   [https://dzul-ikram.github.io/UTS-ThreatHunting/](https://dzul-ikram.github.io/UTS-ThreatHunting/)

4. **Laporan Resmi PDF:**  
   `Laporan_Threat_Hunting_Ujian.pdf` (tersedia di repository)